##### **Import the functions**

In [1]:
from pyspark.sql import functions as F
print("Silver notebook started")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 3, Finished, Available, Finished, False)

Silver notebook started


##### **Read the Bronze tables**

In [2]:
customers = spark.read.table("bronze_customers")
products = spark.read.table("bronze_products")
orders = spark.read.table("bronze_orders")

print("Bronze tables loaded")

print("Customers:", customers.count())
print("Products:", products.count())
print("Orders:", orders.count())

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 4, Finished, Available, Finished, False)

Bronze tables loaded
Customers: 1100
Products: 1063
Orders: 3210


##### **Remove exact duplicate records**

In [3]:
def remove_duplicates(df):
    original_columns = [
        c for c in df.columns if not c.startswith("_")
    ]

    return df.dropDuplicates(original_columns)

customers = remove_duplicates(customers)
products = remove_duplicates(products)
orders = remove_duplicates(orders)

print("Exact duplicates removed")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 5, Finished, Available, Finished, False)

Exact duplicates removed


##### **Clean text columns**

In [4]:
def clean_text(df):
    for column_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column_name,
                F.when(
                    F.trim(F.col(column_name)) == "",
                    None
                ).otherwise(F.trim(F.col(column_name)))
            )

    return df

customers = clean_text(customers)
products = clean_text(products)
orders = clean_text(orders)

print("Text cleaning completed")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 6, Finished, Available, Finished, False)

Text cleaning completed


##### **Clean customer data**

In [5]:
# Standardize customer names and cities
customers = customers.withColumn(
    "customer_name", F.initcap(F.col("customer_name"))
)

customers = customers.withColumn(
    "city", F.initcap(F.col("city"))
)

# Convert email to lowercase
customers = customers.withColumn(
    "email", F.lower(F.col("email"))
)

# Check whether email is valid
customers = customers.withColumn(
    "is_email_valid",
    F.coalesce(
        F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"),
        F.lit(False)
    )
)

# Set invalid email addresses to null
customers = customers.withColumn(
    "email",
    F.when(F.col("is_email_valid"), F.col("email"))
    .otherwise(None)
)

# Keep digits only in phone number
customers = customers.withColumn(
    "phone",
    F.regexp_replace(F.col("phone").cast("string"), r"\D", "")
)

# Remove India's 91 prefix if the number has 12 digits
customers = customers.withColumn(
    "phone",
    F.when(
        (F.length("phone") == 12) &
        F.col("phone").startswith("91"),
        F.substring("phone", 3, 10)
    ).otherwise(F.col("phone"))
)

# Keep only 10-digit phone numbers
customers = customers.withColumn(
    "phone",
    F.when(F.length("phone") == 10, F.col("phone"))
    .otherwise(None)
)

# Standardize state names and state codes
customers = customers.withColumn(
    "state",
    F.when(F.upper("state").isin("MH", "MAHARASHTRA"), "Maharashtra")
    .when(F.upper("state").isin("KA", "KARNATAKA"), "Karnataka")
    .when(F.upper("state").isin("TN", "TAMIL NADU"), "Tamil Nadu")
    .when(F.upper("state").isin("GJ", "GUJARAT"), "Gujarat")
    .when(F.upper("state").isin("DL", "DELHI"), "Delhi")
    .when(F.upper("state").isin("TS", "TELANGANA"), "Telangana")
    .otherwise(F.col("state"))
)

# Parse signup dates in the three accepted formats
customers = customers.withColumn(
    "signup_date",
    F.coalesce(
        F.to_date("signup_date", "yyyy-MM-dd"),
        F.to_date("signup_date", "dd/MM/yyyy"),
        F.to_date("signup_date", "dd-MMM-yyyy")
    )
)

print("Customer cleaning completed")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 7, Finished, Available, Finished, False)

Customer cleaning completed


##### **Clean product data**

In [6]:
# Standardize product category
products = products.withColumn(
    "category", F.initcap(F.col("category"))
)

# Remove currency symbols, Rs, commas and spaces
products = products.withColumn(
    "unit_price",
    F.regexp_replace(
        F.col("unit_price").cast("string"),
        r"(?i)₹|rs\.?|,|\s",
        ""
    ).cast("double")
)

print("Product cleaning completed")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 8, Finished, Available, Finished, False)

Product cleaning completed


##### **Clean order data**

In [7]:
# Standardize order status
orders = orders.withColumn(
    "status", F.initcap(F.lower(F.col("status")))
)

# Standardize payment methods
orders = orders.withColumn(
    "payment_mode",
    F.when(F.lower("payment_mode") == "upi", "UPI")
    .when(
        F.lower("payment_mode").isin(
            "card", "credit card", "debit card"
        ),
        "Card"
    )
    .when(
        F.lower("payment_mode").isin(
            "cod", "cash on delivery"
        ),
        "COD"
    )
    .when(
        F.lower("payment_mode").isin(
            "netbanking", "net banking"
        ),
        "NetBanking"
    )
    .otherwise(F.col("payment_mode"))
)

# Parse order date
orders = orders.withColumn(
    "order_date",
    F.coalesce(
        F.to_date("order_date", "yyyy-MM-dd"),
        F.to_date("order_date", "dd/MM/yyyy"),
        F.to_date("order_date", "dd-MMM-yyyy")
    )
)

# Convert quantity to integer
orders = orders.withColumn(
    "quantity", F.col("quantity").cast("integer")
)

# Convert discount to number and replace missing values with 0
orders = orders.withColumn(
    "discount_pct",
    F.coalesce(
        F.col("discount_pct").cast("double"),
        F.lit(0.0)
    )
)

print("Order cleaning completed")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 9, Finished, Available, Finished, False)

Order cleaning completed


##### **Create a place to store rejected records**

In [8]:
rejects = []

def add_rejects(df, source_name, key_column, reason):
    rejected = df.select(
        F.lit(source_name).alias("source_table"),
        F.col(key_column).cast("string").alias("record_key"),
        F.lit(reason).alias("reject_reason"),
        F.current_timestamp().alias("reject_ts")
    )

    rejects.append(rejected)

print("Reject handling is ready")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 10, Finished, Available, Finished, False)

Reject handling is ready


##### **Reject records with missing IDs**

In [9]:
def split_missing_keys(df, key_column, source_name):
    missing = df.filter(
        F.col(key_column).isNull() |
        (F.trim(F.col(key_column).cast("string")) == "")
    )

    valid = df.filter(
        F.col(key_column).isNotNull() &
        (F.trim(F.col(key_column).cast("string")) != "")
    )

    rejected = missing.select(
        F.lit(source_name).alias("source_table"),
        F.lit(None).cast("string").alias("record_key"),
        F.lit("MISSING_" + key_column.upper()).alias("reject_reason"),
        F.current_timestamp().alias("reject_ts")
    )

    rejects.append(rejected)
    return valid

customers = split_missing_keys(
    customers, "customer_id", "customers"
)

products = split_missing_keys(
    products, "product_id", "products"
)

orders = split_missing_keys(
    orders, "order_id", "orders"
)

print("Missing IDs checked")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 11, Finished, Available, Finished, False)

Missing IDs checked


##### **Keep the latest record for each ID**

In [10]:
from pyspark.sql.window import Window

def keep_latest(df, key_column):
    if "updated_at" not in df.columns:
        raise ValueError(
            "The updated_at column is missing. Check the source data."
        )

    window = Window.partitionBy(key_column).orderBy(
        F.col("updated_at").desc_nulls_last()
    )

    return (
        df.withColumn("_row_number", F.row_number().over(window))
          .filter(F.col("_row_number") == 1)
          .drop("_row_number")
    )

customers = keep_latest(customers, "customer_id")
products = keep_latest(products, "product_id")
orders = keep_latest(orders, "order_id")

print("Latest record per ID retained")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 12, Finished, Available, Finished, False)

Latest record per ID retained


##### **Validate product prices**

In [11]:
invalid_products = products.filter(
    F.col("unit_price").isNull() |
    (F.col("unit_price") <= 0)
)

add_rejects(
    invalid_products,
    "products",
    "product_id",
    "INVALID_PRICE"
)

products = products.filter(
    F.col("unit_price").isNotNull() &
    (F.col("unit_price") > 0)
)

print("Product price validation completed")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 13, Finished, Available, Finished, False)

Product price validation completed


##### **Save customers and products to Silver**

In [12]:
customers.write.format("delta").mode("overwrite").saveAsTable(
    "silver_customers"
)

products.write.format("delta").mode("overwrite").saveAsTable(
    "silver_products"
)

print("Silver customers and products saved")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 14, Finished, Available, Finished, False)

Silver customers and products saved


##### **Validate orders**

In [13]:
# Get the valid customer IDs
customer_ids = (
    customers.select("customer_id").distinct()
    .withColumn("_customer_found", F.lit(True))
)

# Get the valid product IDs
product_ids = (
    products.select("product_id").distinct()
    .withColumn("_product_found", F.lit(True))
)

# Check whether each order's customer and product exist
checked_orders = (
    orders
    .join(customer_ids, on="customer_id", how="left")
    .join(product_ids, on="product_id", how="left")
)

# Assign only the first failing reason
checked_orders = checked_orders.withColumn(
    "_reject_reason",
    F.when(
        F.col("order_date").isNull(),
        "INVALID_ORDER_DATE"
    )
    .when(
        F.col("order_date") > F.lit("2026-10-31").cast("date"),
        "FUTURE_ORDER_DATE"
    )
    .when(
        F.col("customer_id").isNull() |
        (F.trim(F.col("customer_id").cast("string")) == ""),
        "MISSING_CUSTOMER_ID"
    )
    .when(
        F.col("quantity").isNull() | (F.col("quantity") <= 0),
        "INVALID_QUANTITY"
    )
    .when(
        (F.col("discount_pct") < 0) |
        (F.col("discount_pct") > 100),
        "INVALID_DISCOUNT"
    )
    .when(
        F.col("_customer_found").isNull(),
        "UNKNOWN_CUSTOMER"
    )
    .when(
        F.col("_product_found").isNull(),
        "UNKNOWN_PRODUCT"
    )
    .otherwise(None)
)

print("Order validation completed")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 15, Finished, Available, Finished, False)

Order validation completed


##### **Separate valid and rejected orders**

In [14]:
# Orders that passed every validation
valid_orders = checked_orders.filter(
    F.col("_reject_reason").isNull()
)

# Orders that failed a validation
invalid_orders = checked_orders.filter(
    F.col("_reject_reason").isNotNull()
)

# Store rejected orders with their actual failure reason
order_rejects = invalid_orders.select(
    F.lit("orders").alias("source_table"),
    F.col("order_id").cast("string").alias("record_key"),
    F.col("_reject_reason").alias("reject_reason"),
    F.current_timestamp().alias("reject_ts")
)

rejects.append(order_rejects)

# Remove helper columns before saving Silver orders
valid_orders = valid_orders.drop(
    "_customer_found",
    "_product_found",
    "_reject_reason"
)

print("Valid orders:", valid_orders.count())
print("Rejected orders:", invalid_orders.count())

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 16, Finished, Available, Finished, False)

Valid orders: 2805
Rejected orders: 195


##### **Save Silver orders and rejected records**

In [15]:
# Save valid orders
valid_orders.write.format("delta").mode("overwrite").saveAsTable(
    "silver_orders"
)

# Combine rejected records from customers, products and orders
silver_rejects = rejects[0]

for reject_df in rejects[1:]:
    silver_rejects = silver_rejects.unionByName(reject_df)

silver_rejects.write.format("delta").mode("overwrite").saveAsTable(
    "silver_rejects"
)

print("Silver orders and rejects saved")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 17, Finished, Available, Finished, False)

Silver orders and rejects saved


##### **Record the Silver run in the control log**

In [16]:
# Use "manual" for testing.
# In the pipeline, pass the pipeline Run ID through a notebook parameter.
run_id = "manual"

log_data = [
    (run_id, "Silver", "silver_customers",
     customers.count(), "SUCCESS"),
    (run_id, "Silver", "silver_products",
     products.count(), "SUCCESS"),
    (run_id, "Silver", "silver_orders",
     valid_orders.count(), "SUCCESS"),
    (run_id, "Silver", "silver_rejects",
     silver_rejects.count(), "SUCCESS")
]

log_df = spark.createDataFrame(
    log_data,
    ["run_id", "layer", "table_name", "rows_written", "status"]
)

log_df = log_df.withColumn("log_ts", F.current_timestamp())

log_df.write.format("delta").mode("append").saveAsTable(
    "ctl_run_log"
)

print("Silver run logged successfully")

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 18, Finished, Available, Finished, False)

Silver run logged successfully


##### **Verify Silver tables**

In [17]:
print("Silver Customers")
display(spark.read.table("silver_customers"))

print("Silver Products")
display(spark.read.table("silver_products"))

print("Silver Orders")
display(spark.read.table("silver_orders"))

print("Silver Rejects")
display(spark.read.table("silver_rejects"))

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 19, Finished, Available, Finished, False)

Silver Customers


SynapseWidget(Synapse.DataFrame, f3d26b89-ed11-4d39-8570-9f0a0a69075d)

Silver Products


SynapseWidget(Synapse.DataFrame, 5ebfad70-a1fe-4ffb-b32b-eeb0ce278cb7)

Silver Orders


SynapseWidget(Synapse.DataFrame, 5ea9ad2e-5ca5-4376-9799-d63696b68fe7)

Silver Rejects


SynapseWidget(Synapse.DataFrame, 60fba8f6-9282-485a-b2a3-5ac13a24463e)

##### **How many rows**

In [18]:
print("Customers:", spark.read.table("silver_customers").count())
print("Products:", spark.read.table("silver_products").count())
print("Orders:", spark.read.table("silver_orders").count())
print("Rejects:", spark.read.table("silver_rejects").count())

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 20, Finished, Available, Finished, False)

Customers: 1000
Products: 976
Orders: 2805
Rejects: 242


In [19]:
display(
    spark.read.table("silver_rejects")
    .groupBy("source_table", "reject_reason")
    .count()
    .orderBy("source_table", "reject_reason")
)

StatementMeta(, e5088b93-acca-4848-8772-24a17fa9499b, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2b8c1da2-e1b4-460c-bc7c-ec031842d6a4)